In [0]:
from pyspark.sql.functions import (
    col, initcap, regexp_replace, to_date,lower,round,coalesce, 
    current_timestamp, expr, trim, substring,length, row_number,when
)
import re

spark.sql("USE CATALOG 02_silver")
spark.sql("USE SCHEMA default")

def clean_column(name):
    name = re.sub(r'[^a-zA-Z0-9]', '', name)
    return name.lower()

**Cleaning the products table**

In [0]:
df = spark.read.table("01_bronze.default.mst_product_master")
df = df.select([col(c).alias(clean_column(c)) for c in df.columns])

df_products = df.select(
    col("productid").alias("product_id"),

    initcap(
        trim(regexp_replace(col("itemnamedescription"), "[^a-zA-Z\\s]", ""))
    ).alias("product_name"),

    initcap(col("categorytype")).alias("category"),

    regexp_replace(col("basecost"), "[^0-9.]", "").cast("double").alias("cost_price"),

    regexp_replace(col("markedprice"), "[^0-9.]", "").cast("double").alias("marked_price")
)

df_products = df_products.fillna({
    "product_id": "unknown",
    "product_name": "Unknown",
    "category": "Unknown",
    "cost_price": 0,
    "marked_price": 0
})

df_products = df_products.dropDuplicates(["product_id"])

spark.sql("DROP TABLE IF EXISTS 02_silver.default.products")
df_products.write.mode("overwrite").saveAsTable("02_silver.default.products")

**Cleaning the stores table**

In [0]:
df = spark.read.table("01_bronze.default.stores_geo_list_final")

df = df.select([col(c).alias(clean_column(c)) for c in df.columns])

df_stores = df.select(
    col("stid").alias("store_id"),
    
    initcap(
        trim(regexp_replace(col("locationnameaddress"), "[^a-zA-Z\\s]", ""))
    ).alias("store_name"),
    
    initcap(col("cityregion")).alias("city"),
    
    regexp_replace(col("mgrcontact"), "[^0-9]", "").alias("manager_contact")
)

df_stores = df_stores.fillna({
    "store_id": "unknown",
    "store_name": "Unknown",
    "city": "Unknown",
    "manager_contact": "0000000000"
})

df_stores = df_stores.dropDuplicates(["store_id"])

df_stores = df_stores.withColumn("ingested_at", current_timestamp())

spark.sql("DROP TABLE IF EXISTS stores")

df_stores.write.format("delta").mode("overwrite").saveAsTable("stores")